# 06e · Mejora del modelo propio: ensamble de semillas y riesgo de intervención

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Mejorar la Bi-LSTM propia para cumplir los criterios de `docs/metricas.md` que quedaron abiertos en el notebook 07 (C1 y C2 a 60 días, C3 a 180 días, C4), **sin usar el período de validación** para elegir los cambios. |
| **Entradas** | `data/gold/series_filtradas.parquet` (solo train), `data/gold/escaladores.json`, `models/bi_lstm_baseline.pt` (épocas elegidas en el 06) |
| **Salidas** | `models/bi_lstm_v2.pt` (ensamble de 5 semillas + corrección por riesgo de intervención), `reports/modelado/cv_*.csv` |

**Por qué estos cambios.** En la validación del notebook 07, la Bi-LSTM ya es la mejor en los pronósticos que **no** cruzan una intervención de mantenimiento (RMSE a 60 días 0,0331 V frente a 0,0357 V de la persistencia). Todo su déficit viene del 2,7 % de objetivos que **cruzan** un salto de nivel: el modelo se entrena con la máscara de saltos y proyecta la trayectoria sin intervenciones, mientras que la intervención baja el voltaje ~0,5 V. Se prueban dos cambios:
1. **Ensamble de 5 semillas:** promediar 5 entrenamientos idénticos con distinta semilla reduce la varianza del modelo.
2. **Corrección por riesgo de intervención:** sumar la esperanza histórica de los saltos según el percentil de voltaje de la celda, atenuada por un factor λ.

**Protocolo (sin mirar validación).** Validación cruzada temporal con 3 bloques de train. En cada bloque el modelo se entrena solo con orígenes y objetivos anteriores al bloque, y se evalúa como en el notebook 07: VMD causal, voltaje real sin filtrar, mismas métricas. λ se elige con una regla fija y la validación se usa una sola vez, en el notebook 07.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice")

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed

from voltcast import cleaning as cl
from voltcast import config, io, vmd
from voltcast import dataset as ds
from voltcast import metrics as mt
from voltcast import model as mdl
from voltcast import riesgo as rg

config.ensure_dirs()
OUT = config.REPORTS_DIR / "modelado"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = config.PROPIO_SEEDS
_, ck_base = mdl.load_checkpoint(config.MODELS_DIR / "bi_lstm_baseline.pt", "cpu")
EPOCAS = ck_base["mejor_epoca"]  # épocas elegidas por early stopping en el notebook 06
print(f"Dispositivo: {DEVICE} | semillas {SEEDS} | épocas por entrenamiento: {EPOCAS}")

## 1. Datos de train

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
esc = io.read_json("gold", "escaladores.json")
train = gold[gold["split"] == "train"]
CELLS = sorted(train["celda"].astype(str).unique())
data = ds.SeriesTensors(train, CELLS, esc["sigma_delta"], DEVICE)
FECHAS, T_ = data.dates, len(data.dates)
L, H, C = config.LOOKBACK_DAYS, config.HORIZON_DAYS, len(CELLS)
V = ds.pivot(train, "voltaje", CELLS).reindex(FECHAS)
Q = ds.pivot(train, "calidad", CELLS).reindex(FECHAS)
VALIDO = Q.isin(cl.CALIDAD_VALIDA).to_numpy()
OOS = (Q == "fuera_servicio").to_numpy()
VR = V.to_numpy()
VRV = np.where(VALIDO, VR, np.nan)
KA, VC = (ds.pivot(train, c, CELLS).reindex(FECHAS).to_numpy() for c in ("kA_z", "vcel_z"))
saltos = cl.level_shifts(V, Q.apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes))
data.set_level_shifts(saltos)
EL_FLAG = np.array([float(c[0] == "B") for c in CELLS])
print(f"{C} celdas × {T_} días de train ({FECHAS[0].date()} → {FECHAS[-1].date()}) | saltos de nivel en train: {len(saltos)}")

## 2. ¿Las intervenciones son predecibles?

Para cada salto de nivel de train se mide el percentil de voltaje de la celda dentro de su electrolizador un mes antes del salto (media de los días −37 a −7).

In [ ]:
VV = V.where(Q.isin(cl.CALIDAD_VALIDA))
pct_previo = []
for f, c in saltos[["fecha", "celda"]].itertuples(index=False):
    t = FECHAS.get_loc(pd.Timestamp(f))
    el = [x for x in CELLS if x[0] == c[0]]
    pct_previo.append(VV.iloc[max(0, t - 37):t - 7][el].mean().rank(pct=True)[c])
saltos_an = saltos.assign(percentil_previo=pct_previo, anio=pd.to_datetime(saltos["fecha"]).dt.year)
tabla = (saltos_an.assign(rango=pd.cut(saltos_an["percentil_previo"], rg.BINS, right=False))
         .groupby("rango", observed=False).agg(saltos=("delta_V", "size"), delta_V_medio=("delta_V", "mean")))
tabla["% de los saltos"] = 100 * tabla["saltos"] / tabla["saltos"].sum()
tabla.to_csv(OUT / "cv_saltos_por_percentil.csv")
print("Saltos por año:", saltos_an.groupby("anio").size().to_dict())
tabla.round(3)

**Lectura.** Las intervenciones **no son al azar**: el 67 % de los 368 saltos de train ocurrió en celdas que estaban en el 10 % de mayor voltaje de su electrolizador, y el 41 % en el 5 % superior, con caídas medias de −0,49 a −0,58 V. En las celdas por debajo de la mediana los pocos saltos son subidas (+0,14 V de media). Se sabe **qué** celdas tienen más riesgo, pero no **cuándo**: las campañas son irregulares (112 saltos en 2020, 24 en 2024).

## 3. Validación cruzada temporal (3 bloques de train)

| Bloque | Orígenes | Por qué |
|---|---|---|
| F1 | 2022-07-01 → 2023-06-30 | Período con campañas de mantenimiento (oct-2022, feb-2023) |
| F2 | 2023-07-01 → 2024-07-31 | Campañas de sep-2023 y jul-2024 |
| F3 | últimos 270 días de train | El bloque interno del notebook 06; casi sin intervenciones |

Orígenes cada `EVAL_STRIDE` días; objetivos hasta el fin de train. En cada bloque, los modelos se entrenan con orígenes y objetivos anteriores al primer origen del bloque.

In [ ]:
PARAMS = {el: (v["K"], v["alpha"]) for el, v in esc["vmd_params"].items()}
BLOQUES = {"F1": ("2022-07-01", "2023-06-30"), "F2": ("2023-07-01", "2024-07-31"),
           "F3": (str(FECHAS[T_ - config.ES_BLOCK_DAYS - 1].date()), str(FECHAS[T_ - 2].date()))}
NXT = ds.next_jump_index(FECHAS, CELLS, saltos)


def preparar_bloque(ini, fin):
    i0, i1 = FECHAS.get_loc(pd.Timestamp(ini)), FECHAS.get_loc(pd.Timestamp(fin))
    p0 = np.arange(i0, i1 + 1, config.EVAL_STRIDE)

    def celda(j):
        K, a = PARAMS[CELLS[j][0]]
        return np.stack([vmd.causal_window(VR[:, j], OOS[:, j], t0, K, a) for t0 in p0])

    W = np.stack(Parallel(n_jobs=config.N_JOBS)(delayed(celda)(j) for j in range(C)), axis=1)  # [O, C, L]
    win = p0[:, None] + np.arange(-L + 1, 1)
    tgt = p0[:, None] + np.arange(1, H + 1)
    tgt_c = np.minimum(tgt, T_ - 1)
    REAL = VR[tgt_c].transpose(0, 2, 1)
    WR = VRV[win].transpose(0, 2, 1)
    ok = np.isfinite(W).all(-1) & ~OOS[win].transpose(0, 2, 1).any(-1) & np.isfinite(WR).any(-1)
    MASK = VALIDO[tgt_c].transpose(0, 2, 1) & (tgt <= T_ - 1)[:, None, :] & ok[..., None] & np.isfinite(REAL)
    return dict(i0=i0, p0=p0, win=win, W=W, WR=WR, REAL=REAL, MASK=MASK,
                CRUZA=NXT[p0][:, :, None] <= tgt[:, None, :])


def entrenar(seed, hasta):
    """Bi-LSTM con la configuración del notebook 06, entrenada con orígenes y objetivos ≤ hasta."""
    torch.manual_seed(seed)
    m = mdl.build_model(C, len(ds.FEATURES), config.TRAIN, use_noa=False).to(DEVICE)
    mdl.fit(m, data, (L - 1, hasta), hasta, config.TRAIN, epochs=EPOCAS, seed=seed, log=None)
    return m


def predecir(m, b):
    out = np.empty((len(b["p0"]), C, H))
    for i, w in enumerate(b["win"]):
        out[i], _ = mdl.predict_volts(m, b["W"][i], VALIDO[w].T, KA[w].T, VC[w].T, esc["sigma_delta"],
                                      np.arange(C), EL_FLAG, DEVICE)
    return out


CV = {}
for nombre, (ini, fin) in BLOQUES.items():
    t = time.time()
    b = preparar_bloque(ini, fin)
    b["pred"] = {s: predecir(entrenar(s, b["i0"] - 1), b) for s in SEEDS}
    CV[nombre] = b
    print(f"{nombre}: {len(b['p0'])} orígenes ({ini} → {fin}) | objetivos {b['MASK'].sum():,} | "
          f"cruzan un salto {(b['MASK'] & b['CRUZA']).sum() / b['MASK'].sum():.1%} | {time.time() - t:.0f} s")

### 3.1 Ensamble de semillas

In [ ]:
def metricas(b, p, mask=None):
    mk = b["MASK"] if mask is None else b["MASK"] & mask
    ref = mt.last_observed(b["WR"])
    fila = {"MAE (V)": mt.mae(p, b["REAL"], mk), "RMSE (V)": mt.rmse(p, b["REAL"], mk),
            "R2_cambio": mt.r2_change(p, b["REAL"], ref, mk)}
    for h in config.EVAL_HORIZONS:
        k = h - 1
        fila[f"RMSE {h} d (V)"] = mt.rmse(p[..., k], b["REAL"][..., k], mk[..., k])
        fila[f"R2_cambio {h} d"] = mt.r2_change(p[..., k], b["REAL"][..., k], ref, mk[..., k])
        fila[f"% celdas RMSE {h} d < 0,15 V"] = float(np.nanmean(mt.rmse(p[..., k], b["REAL"][..., k], mk[..., k], axis=0) < 0.15) * 100)
    return fila


filas = []
for nombre, b in CV.items():
    b["ens"] = np.mean([b["pred"][s] for s in SEEDS], axis=0)
    b["per"] = mt.persistence(mt.last_observed(b["WR"]), H)
    filas.append({"bloque": nombre, "modelo": "Persistencia", **metricas(b, b["per"])})
    for s in SEEDS:
        filas.append({"bloque": nombre, "modelo": f"Bi-LSTM semilla {s}", **metricas(b, b["pred"][s])})
    filas.append({"bloque": nombre, "modelo": f"Bi-LSTM ensamble de {len(SEEDS)}", **metricas(b, b["ens"])})
cv_ens = pd.DataFrame(filas).set_index(["bloque", "modelo"])
cv_ens.to_csv(OUT / "cv_ensamble_semillas.csv")
cv_ens[["MAE (V)", "RMSE (V)", "RMSE 60 d (V)", "RMSE 180 d (V)", "R2_cambio"]].round(4)

**Lectura.**
- El RMSE de la Bi-LSTM **varía ±2 % entre semillas** (p. ej. F1: 0,146–0,153 V). Las diferencias pequeñas entre modelos de una sola semilla no son concluyentes.
- El **ensamble de 5 semillas** queda en todos los bloques en o cerca de la mejor semilla, sin que haya que elegirla: es una mejora pequeña pero segura (F3: RMSE 0,0169 V, R² cambio 0,23).
- En F1 (6,4 % de objetivos que cruzan una intervención) ninguna variante supera a la persistencia en RMSE: el mismo problema que en validación.

### 3.2 Corrección por riesgo de intervención

En cada bloque, la esperanza `c[h, percentil]` de la suma de saltos en `(t0, t0 + h]` se estima solo con la historia anterior al bloque (`riesgo.ajustar`). Se prueba `V̂ = V̂_ensamble + λ · c` para varios λ.

**Regla de elección (fijada antes de ver los resultados):** el λ que minimiza el **promedio entre bloques del RMSE relativo** (RMSE con λ ÷ RMSE con λ = 0) sobre el horizonte completo. Se usa el RMSE relativo para que cada bloque pese lo mismo; con el MSE combinado, F1 (errores 5 veces más grandes) decidiría solo.

In [ ]:
PCT = rg.percentil(VRV, CELLS)
ACUM = rg.saltos_acumulados(FECHAS, CELLS, saltos)
LAMBDAS = [0.0, 0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.5, 1.0]
filas = []
for nombre, b in CV.items():
    b["c"] = rg.ajustar(PCT, ACUM, b["i0"] - 1, H)
    b["pct0"] = PCT[b["p0"]]
    r0 = mt.rmse(b["ens"], b["REAL"], b["MASK"])
    r0_60 = mt.rmse(b["ens"][..., 59], b["REAL"][..., 59], b["MASK"][..., 59])
    for lam in LAMBDAS:
        p = rg.aplicar(b["ens"], b["pct0"], b["c"], lam)
        filas.append({"bloque": nombre, "λ": lam, "RMSE relativo 1–180": mt.rmse(p, b["REAL"], b["MASK"]) / r0,
                      "RMSE relativo 60 d": mt.rmse(p[..., 59], b["REAL"][..., 59], b["MASK"][..., 59]) / r0_60})
cv_lam = pd.DataFrame(filas)
cv_lam.to_csv(OUT / "cv_lambda_riesgo.csv", index=False)
prom = cv_lam.groupby("λ")[["RMSE relativo 1–180", "RMSE relativo 60 d"]].mean()
LAMBDA = float(prom["RMSE relativo 1–180"].idxmin())
print(f"λ elegido = {LAMBDA}")
cv_lam.pivot(index="λ", columns="bloque", values="RMSE relativo 1–180").join(prom).round(4)

In [ ]:
filas = []
for nombre, b in CV.items():
    b["v2"] = rg.aplicar(b["ens"], b["pct0"], b["c"], LAMBDA)
    for etiqueta, mk in [("todos", None), ("cruza salto", b["CRUZA"]), ("sin salto", ~b["CRUZA"])]:
        for modelo, p in [("Persistencia", b["per"]), ("Bi-LSTM (semilla 42, como el 06)", b["pred"][SEEDS[0]]),
                          (f"Bi-LSTM v2 (ensamble + riesgo λ = {LAMBDA})", b["v2"])]:
            filas.append({"bloque": nombre, "subconjunto": etiqueta, "modelo": modelo, **metricas(b, p, mk)})
cv_v2 = pd.DataFrame(filas).set_index(["subconjunto", "bloque", "modelo"])
cv_v2.to_csv(OUT / "cv_modelo_v2.csv")
cv_v2.loc["todos"].round(4)

In [ ]:
cv_v2.loc[["cruza salto", "sin salto"], ["MAE (V)", "RMSE (V)", "RMSE 60 d (V)", "RMSE 180 d (V)"]].round(4)

**Lectura.**
- La corrección completa (λ = 1) mejora mucho F1, el bloque con campañas, pero **multiplica el error** en F2 y F3, donde hubo pocas intervenciones: acierta en qué celdas, pero no en cuándo.
- La regla elige **λ = 0,05**: −2,3 % y −3,2 % de RMSE en F1 y F2, y +0,3 % en F3.
- **Bi-LSTM v2 (ensamble + λ = 0,05) supera a la persistencia en RMSE 1–180 en los 3 bloques** (F1 0,1445 frente a 0,1453 V; F2 0,0437 frente a 0,0451; F3 0,0170 frente a 0,0203), algo que la Bi-LSTM original no lograba en F1 ni en F2. A 60 días empata en F1 (0,1231 frente a 0,1230) y gana en F2 y F3.
- La mejora viene de los objetivos que cruzan una intervención (F2: 0,2632 frente a 0,2755 V) a costa de un leve empeoramiento sin salto en F1. Las intervenciones siguen siendo el error dominante.

## 4. Modelo final: entrenamiento con todo train

Mismo procedimiento que el notebook 06 (reentrenamiento con todo train durante las épocas elegidas), con las 5 semillas. La corrección `c` se estima con toda la historia de train.

In [ ]:
t = time.time()
modelos = [entrenar(s, T_ - 1) for s in SEEDS]
ens = mdl.Ensemble(modelos).eval()
C_FINAL = rg.ajustar(PCT, ACUM, T_ - 1, H)
seg = time.time() - t
split = gold.loc[gold["split"] == "val", "fecha"].min()
meta = {k: ck_base[k] for k in ("cells", "features", "train_cfg", "lookback", "horizon", "anchor_days", "escaladores",
                                "vmd_params", "mask_post_jump", "refit_full_train", "mejor_epoca", "entrenado_hasta")}
meta.update({"nombre": "Bi-LSTM v2 (ensamble de semillas + riesgo de intervención)", "use_noa": False,
             "n_modelos": len(SEEDS), "seeds": SEEDS, "riesgo_lambda": LAMBDA, "riesgo_c": C_FINAL.tolist(),
             "riesgo_bins": rg.BINS.tolist(), "tiempo_min": {"entrenamiento_min": seg / 60},
             "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds")})
path = config.MODELS_DIR / "bi_lstm_v2.pt"
mdl.save_checkpoint(path, ens, meta)
print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.2f} MB) | {seg / 60:.1f} min")
pd.DataFrame(C_FINAL[[29, 59, 119, 179]], index=["h = 30", "h = 60", "h = 120", "h = 180"],
             columns=[f"percentil {a:.2f}–{min(b, 1):.2f}" for a, b in zip(rg.BINS[:-1], rg.BINS[1:])]).round(3)

## 5. Verificaciones

In [ ]:
checks = {}
# riesgo.ajustar con saltos sintéticos: una celda que siempre salta −0,5 V el día 100 → c = −0,5 para h ≥ distancia
s_sint = pd.DataFrame({"fecha": [FECHAS[100]], "celda": [CELLS[0]], "delta_V": [-0.5]})
a = rg.saltos_acumulados(FECHAS[:200], CELLS[:1], s_sint)
checks["saltos acumulados: −0,5 V desde el día 100"] = a[99, 0] == 0 and a[100, 0] == -0.5
checks["aplicar con λ = 0 no cambia el pronóstico"] = np.array_equal(rg.aplicar(CV["F3"]["ens"], CV["F3"]["pct0"], C_FINAL, 0.0), CV["F3"]["ens"])
m, ck = mdl.load_checkpoint(path, DEVICE)
x, y, mask, cell, el = data.batch(*data.origins(T_ - 300, T_ - 2, stride=30), T_ - 1)
with torch.no_grad():
    checks["el checkpoint reproduce el ensamble"] = torch.allclose(m(x, cell, el), ens(x, cell, el), atol=1e-5)
checks["solo se usaron fechas de train"] = FECHAS[-1] < split
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se construyó la **Bi-LSTM v2**: ensamble de 5 semillas de la Bi-LSTM del notebook 06 + corrección por riesgo de intervención con λ = 0,05, todo elegido con validación cruzada temporal **dentro de train** (`models/bi_lstm_v2.pt`).
- En la validación cruzada mejora a la Bi-LSTM original y supera a la persistencia en los 3 bloques. La mejora esperada es pequeña (~2 % de RMSE), porque el error dominante son las intervenciones de mantenimiento, cuyo momento no se puede anticipar con los datos disponibles.
- **Resultado en validación (notebook 07, evaluado una sola vez):** mejora marginal (RMSE 1–180 0,0580 frente a 0,0585 V; 60 días 0,0576 frente a 0,0582 V), insuficiente para superar a la persistencia a 60 días, y algo peor a 180 días. **No cambia el cumplimiento de los criterios**; se mantiene la Bi-LSTM original como modelo elegido.
- Lo que limita a todos los modelos es la falta del **registro de mantenimientos** (fechas y celdas intervenidas, o al menos el plan de campañas): con él, la corrección por riesgo podría usar el momento de la intervención y no solo qué celdas tienen más riesgo.